In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import from_json, col, regexp_replace, pandas_udf, to_json, concat, lit, array_join
from pyspark.sql.types import ArrayType, DoubleType
import pandas as pd
from typing import Iterator
import os, shutil
from pyspark.ml.linalg import Vectors, VectorUDT

# ============================================================================
# DATA SOURCE CONFIGURATION
# ============================================================================
USE_JSON_FILES = True  # Set to False when switching to Event Hubs

# JSON files configuration (for testing)
JSON_SOURCE_PATH = "/Volumes/main/default/streaming_data/"
SCHEMA_LOCATION = "/Volumes/main/default/streaming_data/_schemas/realtime_stream"

# Event Hubs configuration (for production)
EVENT_HUBS_NAMESPACE = "<your-namespace>"  # e.g. "my-eh-namespace"
EVENT_HUBS_TOPIC = "logs-topic"
EVENT_HUBS_CONNECTION_STRING = "<your-connection-string>"  # Endpoint=sb://...

# Unity Catalog configuration (Lakebase)
CATALOG = "real_time_db"  # Unity Catalog catalog
SCHEMA = "public"  # Unity Catalog schema

# Checkpoint locations
CHECKPOINT_LOGS = "/Volumes/main/default/streaming_data/_checkpoints/standalone_logs"
CHECKPOINT_METRICS = "/Volumes/main/default/streaming_data/_checkpoints/standalone_metrics"

# Schemas
envelope_schema = StructType([
    StructField("value", StringType(), True),
    StructField("key", StringType(), True),
    StructField("topic", StringType(), True),
    StructField("partition", IntegerType(), True),
    StructField("offset", StringType(), True),
    StructField("timestamp", StringType(), True)
])

payload_schema = "timestamp DOUBLE, type STRING, cpu_percent DOUBLE, memory STRUCT<total: BIGINT, available: BIGINT, percent: DOUBLE>, disk_io STRUCT<read_bytes: BIGINT, write_bytes: BIGINT>, category STRING, log_raw STRING"

# UDF for embeddings
@pandas_udf(ArrayType(DoubleType()))
def generate_embedding_udf(iterator: Iterator[pd.Series]) -> Iterator[pd.Series]:
    import mlflow.deployments
    import pandas as pd

    ml_client = mlflow.deployments.get_deploy_client("databricks")

    for texts in iterator:
        if len(texts) == 0:
            yield pd.Series([], dtype=object)
            continue

        text_batch = [
            str(t) if t is not None else ""
            for t in texts.tolist()
        ]

        response = ml_client.predict(
            endpoint="databricks-bge-large-en",
            inputs={"input": text_batch}
        )

        embeddings = [
            [float(x) for x in item["embedding"]]
            for item in response["data"]
        ]

        yield pd.Series(embeddings)

print("🚀 Starting single-hop streaming...")

In [0]:
# ============================================================================
# Stream 1: Logs → Lakebase (with embeddings)
# ============================================================================

print("🚀 Starting logs stream...")
print(f"   Source: {'JSON files' if USE_JSON_FILES else 'Event Hubs'}")

if USE_JSON_FILES:
    # JSON files via Auto Loader (for testing)
    logs_stream = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)
        .option("cloudFiles.maxFilesPerTrigger", "10")
        .schema(envelope_schema)
        .load(JSON_SOURCE_PATH)
        .withColumn("payload", from_json(col("value"), payload_schema))
        .where("payload.type = 'log'")
        .select(
            col("timestamp").alias("eventhub_timestamp"),
            col("topic"),
            col("payload.timestamp").alias("timestamp"),
            col("payload.category").alias("category"),
            regexp_replace(col("payload.log_raw"), '"eventMessage" : ', '').alias("log_raw")
        )
        .withColumn("embedding_vector", generate_embedding_udf(col("log_raw")))
        .select("eventhub_timestamp", "topic", "timestamp", "category", "log_raw", "embedding_vector")
    )
else:
    # Event Hubs via Kafka (for production)
    logs_stream = (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", f"{EVENT_HUBS_NAMESPACE}.servicebus.windows.net:9093")
        .option("subscribe", EVENT_HUBS_TOPIC)
        .option("kafka.sasl.mechanism", "PLAIN")
        .option("kafka.security.protocol", "SASL_SSL")
        .option("kafka.sasl.jaas.config", 
                f'org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{EVENT_HUBS_CONNECTION_STRING}";')
        .option("startingOffsets", "latest")
        .option("maxOffsetsPerTrigger", "100")
        .load()
        .withColumn("payload", from_json(col("value").cast("string"), payload_schema))
        .where("payload.type = 'log'")
        .select(
            col("timestamp").alias("eventhub_timestamp"),
            col("topic"),
            col("payload.timestamp").alias("timestamp"),
            col("payload.category").alias("category"),
            regexp_replace(col("payload.log_raw"), '"eventMessage" : ', '').alias("log_raw")
        )
        .withColumn("embedding_vector", generate_embedding_udf(col("log_raw")))
        .select("eventhub_timestamp", "topic", "timestamp", "category", "log_raw", "embedding_vector")
    )

logs_query = (
    logs_stream.writeStream
    .format("postgresql")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_LOGS)
    .trigger(availableNow=True)
    .toTable(f"{CATALOG}.{SCHEMA}.enriched_logs")
)

print(f"✅ Logs stream started (checkpoint: {CHECKPOINT_LOGS})")

In [0]:
import time
from datetime import datetime

# Note: Checkpoint is preserved to track processed files
# Remove these lines to clean checkpoint and reprocess all data:
# if os.path.exists(CHECKPOINT_METRICS):
#     shutil.rmtree(CHECKPOINT_METRICS)
#     print(f"✅ Cleaned checkpoint: {CHECKPOINT_METRICS}")

print("🔥 Starting continuous metrics streaming (while loop)...")
print(f"   Source: {'JSON files' if USE_JSON_FILES else 'Event Hubs'}")
print(f"   Path: {JSON_SOURCE_PATH if USE_JSON_FILES else EVENT_HUBS_NAMESPACE}")
print(f"   Target: {CATALOG}.{SCHEMA}.parsed_metrics")
print("   Press STOP button to terminate")
print("=" * 80)

iteration = 0

try:
    while True:
        iteration += 1
        timestamp = datetime.now().strftime("%H:%M:%S")
        
        # Stream 2: Metrics → Lakebase
        if USE_JSON_FILES:
            # JSON files via Auto Loader (for testing)
            metrics_stream = (
                spark.readStream
                .format("cloudFiles")
                .option("cloudFiles.format", "json")
                .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)
                .option("cloudFiles.maxFilesPerTrigger", "10")
                .schema(envelope_schema)
                .load(JSON_SOURCE_PATH)
                .withColumn("payload", from_json(col("value"), payload_schema))
                .where("payload.type = 'metrics'")
                .select(
                    col("timestamp").alias("eventhub_timestamp"),
                    col("topic"),
                    col("payload.timestamp").alias("timestamp"),
                    col("payload.cpu_percent").alias("cpu_percent"),
                    col("payload.memory.total").alias("total_memory"),
                    col("payload.memory.available").alias("available_memory"),
                    col("payload.memory.percent").alias("percent_memory")
                )
            )
        else:
            # Event Hubs via Kafka (for production)
            metrics_stream = (
                spark.readStream
                .format("kafka")
                .option("kafka.bootstrap.servers", f"{EVENT_HUBS_NAMESPACE}.servicebus.windows.net:9093")
                .option("subscribe", EVENT_HUBS_TOPIC)
                .option("kafka.sasl.mechanism", "PLAIN")
                .option("kafka.security.protocol", "SASL_SSL")
                .option("kafka.sasl.jaas.config", 
                        f'org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{EVENT_HUBS_CONNECTION_STRING}";')
                .option("startingOffsets", "latest")
                .option("maxOffsetsPerTrigger", "100")
                .load()
                .withColumn("payload", from_json(col("value").cast("string"), payload_schema))
                .where("payload.type = 'metrics'")
                .select(
                    col("timestamp").alias("eventhub_timestamp"),
                    col("topic"),
                    col("payload.timestamp").alias("timestamp"),
                    col("payload.cpu_percent").alias("cpu_percent"),
                    col("payload.memory.total").alias("total_memory"),
                    col("payload.memory.available").alias("available_memory"),
                    col("payload.memory.percent").alias("percent_memory")
                )
            )
        
        # Process one batch
        metrics_query = (
            metrics_stream.writeStream
            .format("postgresql")
            .outputMode("append")
            .option("checkpointLocation", CHECKPOINT_METRICS)
            .trigger(availableNow=True)
            .toTable(f"{CATALOG}.{SCHEMA}.parsed_metrics")
        )
        
        # Wait for this batch to complete
        metrics_query.awaitTermination(timeout=10)
        
        print(f"[{timestamp}] Iteration {iteration:4d} - Batch processed ✅")
        
        # Wait 2 seconds before next iteration
        time.sleep(1)
        
except KeyboardInterrupt:
    print(f"\n\n✅ Stopped after {iteration} iterations")
except Exception as e:
    print(f"\n\n❌ Error: {e}")
    raise